<a href="https://colab.research.google.com/github/u9828057/Python_Practice/blob/main/Electronics_with_Python/Electronics_Simulation_09_03_Python_Based_Semiconductor_Manufacturing_Data_Analysis_Prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("--------(Electronics_Simulation: Python-Based \
Semiconductor Manufacturing Data Analysis Prototype)--------\n")

def generated_simulated_data(
    n_lots=40,
    wafers_per_lot=25,
    seed=214952213,
    output_path="simulated_semiconductor_data.csv"
):

  """
  產生半導體機台模擬數據 (包含 Tool_B 的刻意異常與少量缺失值)

  參數
  ----------
  n_lots : int
    機台生產的批次數量

  wafers_per_lot : int
    各生產批次的晶圓數量

  seed : int
    結果可再現性的隨機種子值

  output_path: str
    CSV 輸出檔案路徑

  回傳
  ----------
  pd.DataFrame
    半導體製程模擬數據集
  """

  # 固定亂數種子，確保結果可以重現
  np.random.seed(seed)
  n_samples = n_lots * wafers_per_lot

  # 建立 Lot ID 與 Wafer ID
  lot_ids = [
      f"L{str(i).zfill(3)}"
      for i in range(1, n_lots + 1)
      for _ in range(wafers_per_lot)
  ]

  wafer_ids = [
      f"W{str(i).zfill(2)}"
      for _ in range(n_lots)
      for i in range(1, wafers_per_lot + 1)
  ]

  # 模擬三台製程設備
  tool_choices = ["Tool_A", "Tool_B", "Tool_C"]

  # 假設同一 Lot 使用同一台 Tool
  lot_tool_map = {
      f"L{str(i).zfill(3)}": np.random.choice(tool_choices)
      for i in range(1, n_lots + 1)
  }

  tools = [lot_tool_map[lot] for lot in lot_ids]

  # 建立基礎 DataFrame
  df = pd.DataFrame({
      "Lot_ID": lot_ids,
      "Wafer_ID": wafer_ids,
      "Tool_ID": tools,
      "Temperature_C": np.zeros(n_samples),
      "Pressure_Torr": np.zeros(n_samples),
      "CD_nm": np.zeros(n_samples),
      "Yield_Percent": np.zeros(n_samples)
  })

  # 根據不同的機台建立不同的製程分布
  for tool in tool_choices:

    mask = df["Tool_ID"] == tool
    n_tool = mask.sum()

    if tool == "Tool_B":

      # 模擬異常設備
      df.loc[mask, "Temperature_C"] = np.random.normal(403.5, 1.2, n_tool)
      df.loc[mask, "Pressure_Torr"] = np.random.normal(5.2, 0.15, n_tool)
      df.loc[mask, "CD_nm"] = np.random.normal(42.5, 0.8, n_tool)
      df.loc[mask, "Yield_Percent"] = np.random.normal(88.5, 2.5, n_tool)

    else:

      # 模擬正常設備
      df.loc[mask, "Temperature_C"] = np.random.normal(400.0, 0.8, n_tool)
      df.loc[mask, "Pressure_Torr"] = np.random.normal(5.0, 0.1, n_tool)
      df.loc[mask, "CD_nm"] = np.random.normal(40.0, 0.4, n_tool)
      df.loc[mask, "Yield_Percent"] = np.random.normal(97.5, 1.2, n_tool)

  # 加入少量缺失值，供 Data Cleaning 練習
  nan_idx = np.random.choice(df.index, 15, replace=False)
  df.loc[nan_idx[:7], "CD_nm"] = np.nan
  df.loc[nan_idx[7:], "Yield_Percent"] = np.nan

  # 良率限制在合理範圍
  df["Yield_Percent"] = df["Yield_Percent"].clip(lower=0, upper=100)

  # 匯出 CSV
  df.to_csv(output_path, index=False)

  return df







--------(Electronics_Simulation: Python-Based Semiconductor Manufacturing Data Analysis Prototype)--------



,Lot_ID,Wafer_ID,Tool_ID,Temperature_C,Pressure_Torr,CD_nm,Yield_Percent
0,L001,W01,Tool_C,399.379984,4.914169,40.443851,97.504508
1,L001,W02,Tool_C,399.179297,4.942768,39.205623,97.104235
2,L001,W03,Tool_C,400.295474,5.090992,40.164290,96.475696
3,L001,W04,Tool_C,399.354299,4.901913,40.744141,96.299321
4,L001,W05,Tool_C,401.754608,4.969747,39.771581,99.521869
...,...,...,...,...,...,...,...
995,L040,W21,Tool_A,400.577186,4.936163,40.072752,97.318129
996,L040,W22,Tool_A,400.605543,5.071860,40.255764,99.040453
997,L040,W23,Tool_A,400.074866,4.787845,39.784308,98.534531
998,L040,W24,Tool_A,400.632702,4.947717,39.816140,NaN


In [ ]:
# § 補充資料

#============================================================
# 模擬 40 個批次 (Lot)，每個批次 25 片晶圓 (Wafer)
#============================================================

# 【Lot ID 產生邏輯】
# 1. str(i).zfill(3): 將批次數字 i 轉成字串，並在左邊補零直到長度為 3 (例如 1 變成 "001")
# 2. for _ in range(25): 底線 _ 代表「無用變數」，這裡單純用來讓迴圈重複 25 次
# 3. 組合效果: 把產生出來的 "L001" 重複放入清單 25 次，再換下一個批次 "L002" 重複 25 次

# lot_ids = [
#    f"L{str(i).zfill(3)}"
#    for i in range(1, n_lots+1)
#    for _ in range(wafers_per_lot)
# ]

#-------------------------------------------------------------

# 【Wafer ID 產生邏輯】
# 1. str(i).zfill(2): 將晶圓數字 i 轉成字串，補零直到長度為 2 (例如 1 變成 "01")
# 2. 外層先跑 for _ in range(n_lots) (重複 40 遍)，內層跑 for i in range(1, wafers_per_lot+1) (產生 1～25)
# 3. 組合效果: 產生 ["W01", "W02",... "W25"] 這樣的循環，並重複 40 組來配對 Lot ID

# wafer_ids = [
#     f"W{str(i).zfill(2)}"
#     for _ in range(n_lots)
#     for i in range(1, wafers_per_lot+1)
# ]


#============================================================================
# 模擬製程設備分配，確保「同一批次 (Lot) 的 25 片晶圓進入同一台機台」
#============================================================================

# 1. 定義廠內可用的機台清單
# tool_choices = ["Tool_A", "Tool_B", "Tool_C"]


# 【建立批次與機台對照表 (Lookup Table / Dictionary)】
# 邏輯說明:
# 1. 使用「字典推導式」，建立一個 Lot ID 對應 Tool 的字典
# 2. 迴圈跑 n_lots (例如 40) 次，每次為一個批次隨機抽籤 (np.random.choice) 分配一台機台
# 3. 產生的結果類似「派工單」: {"L001": "Tool_B", "L002": "Tool_A", ... 總共 40 筆}
# 目的: 綁定「批次」與「機台」的關係，避免同一批晶圓被拆散

# lot_tool_map = {
#     f"L{str(i).zfill(3)}": np.random.choice(tool_choices)
#     for i in range(1, n_lots+1)
# }

#-------------------------------------------------------------

# 【將機台分配結果展開至所有晶圓資料】
# 邏輯說明:
# 1. lot_ids 是一個長度為 1000 的清單 (例如: ["L001", "L001", ... 25次, "L002"...])
# 2. 迴圈會巡視這 1000 個 lot_ids，每次抓出一個值放進變數 lot (例如: lot = "L001")
# 3. lot_tool_map[lot] 語法代表「字典查詢」:
# 把 "L001" 丟進字典查表，字典就會吐出之前綁定的機台 (例如: "Tool_B")
# 4. 組合效果: 確保 "L001" 的 25 片晶圓，每次查表得到的結果絕對一模一樣，比較符合此模擬的製程假設

# tools = [lot_tool_map[lot] for lot in lot_ids]


#====================================================
# 根據不同機台設定具有「製程關聯特徵」的模擬數據
#====================================================

# 迴圈依序處理 Tool_A, Tool_B, Tool_C
# for tool in tool_choices:
    # 1. 【布林遮罩 (Boolean Mask)】
    # mask 會產生一串 True/False，標記出那些列 (Row) 是屬於目前這台機器的
    # n_tool 算出這些 True 的總數 (該機台生產的晶圓總片數)

    # mask = df["Tool_ID"] == tool
    # n_tool = mask.sum()

    # 2. 【產生常態分布數據】與【資料定位 (df.loc)】
    # np.random.normal(平均值, 標準差, 數量): 產生符合常態分布的隨機數
    # df.loc[mask, "欄位名稱"]: 只針對 mask 為 True 的列，修改特定欄位的數值

    # if tool == "Tool_B":
      # 模擬異常設備 (刻意設計的製程關聯特徵):
      # 溫度偏高且不穩定 (平均 403.5, 標準差 1.2) → 導致 CD 線寬變大 → 最終良率變低 (平均 88.5%)
      # df.loc[mask, "Temperature_C"] = np.random.normal(403.5, 1.2, n_tool)
      # ... (其他參數省略)

    # else:
      # 模擬正常設備 (Tool_A, Tool_C):
      # 溫控精準 (平均 400.0) → CD 線寬達標 → 良率極高 (平均 97.5%)
      # df.loc[mask, "Temperature_C"] = np.random.normal(400.0, 0.8, n_tool)
      # ... (其他參數省略)


#=========================================================
# 隨機加入缺失值 (NaN)，模擬真實產線感測器斷線的狀況
#=========================================================

# 1. np.random.choice: 從 1000 筆列索引 (df.index) 中，隨機抽出 15 個

# 2. replace=False: 「取後不放回」，確保抽出的 15 個位置完全不重複
# nan_idx = np.random.choice(df.index, 15, replace=False)

# 3. df.loc[列, 欄]: 座標定位
# loc (location) 語法: 先指定列 (Row)，再指定欄 (Column)

# 4. 利用陣列切片 (Slicing)，將前 7 筆的 CD_nm 設為空值 (np.nan)
# df.loc[nan_idx[:7], "CD_nm"] = np.nan

# 5. 將剩下的 8 筆 (索引 7 到最後) 的 Yield_Percent 設為空值
# df.loc[nan_idx[7:], "Yield_Percent"] = np.nan

# 6. clip: 物理邊界裁切
# 常態分布可能會產生大於 100% 或小於 0% 的良率，這在物理上是不可能的
# clip(lower=0, upper=100) 會把所有超過 100 的數字強制變成 100，低於 0 的變成 0
# df["Yield_Percent"] = df["Yield_Percent"].clip(lower=0, upper=100)


In [ ]:
#=========================================
# --- Colab 畫圖中文顯示起手式 ---
#=========================================
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 下載微軟正黑體 (或是 Noto 繁體中文)
!wget -q -O NotoSansTC-Regular.otf \
https://github.com/googlefonts/noto-cjk/raw/main/Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf
fm.fontManager.addfont('NotoSansTC-Regular.otf')
plt.rc('font', family='Noto Sans CJK TC')
plt.rcParams['axes.unicode_minus'] = False
# --------------------------------

print("環境設定完成！可以開始畫圖了！")